<a href="https://colab.research.google.com/github/CanopySimulations/canopy-python-examples/blob/master/loading_saved_components.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Upgrade Runtime
This cell ensures the runtime supports `asyncio` async/await, and is needed on Google Colab. If the runtime is upgraded, you will be prompted to restart it, which you should do before continuing execution.

In [ ]:
!pip install ipython ipykernel --upgrade

# Set Up Environment

### Import required libraries

In [ ]:
!pip install -q canopy

In [ ]:
import canopy
import logging
import nest_asyncio

logging.basicConfig(level=logging.INFO)
nest_asyncio.apply()

### Authenticate

In [ ]:
authentication_data = canopy.prompt_for_authentication()
session = canopy.Session(authentication_data)
session.authentication.authenticate()

## Introduction: Config Management

You can save and load configs at any level within the car model, allowing you to build up libraries of sub-components. Each one of these sub-components can be named and have tags and notes applied to make them more searchable.
They can also be encrypted and saved, downloaded, exported, and catalogued via the API.

Remember, everything you can do through the UI can be done through the API as well.

## Example 1: List all saved track configurations

We're going to load all of the (non-default) saved track configurations and print a table of their name and their elevation above sea level

In [ ]:
# We don't have a neat wrapper for this function, so we'll use the API directly
from typing import List

config_api = canopy.openapi.ConfigApi(session.async_client)

# Note that these are returned in order of oldest first
metadata_result: canopy.openapi.GetConfigsQueryResult = await config_api.config_get_configs(session.authentication.tenant_id, 'track')

documents: List[canopy.openapi.CanopyDocument] = metadata_result.query_results.documents
if len(documents) == 0:
    raise canopy.NotFoundError('No config found matching the specified criteria.')

async def load_track_from_document(document: canopy.openapi.CanopyDocument):
    tenant_id = session.authentication.tenant_id
    track = await canopy.load_config(session, document.document_id, tenant_id=tenant_id)
    logging.info(f'Loaded track: {track.raw_data['name']} ({track.config_id})')
    return track

# I'm explicitly taking the last five elements here to get the most recent tracks
all_tracks = [await load_track_from_document(doc) for doc in documents[-5:]]



# Build up a data table for the tracks that we've recorded
track_data_table = []

for track in all_tracks:
    # I carefully call raw_data here. If we call data directly, it triggers lazy
    # conversion of the dictionary to a strongly typed object, is slower, and not necessary
    track_data_table.append({
        'Track Name': track.raw_data['name'],
        'Elevation': track.raw_data['hTrackAboveSeaLevel']
    })

# Display the data table
import pandas as pd
df = pd.DataFrame(track_data_table)
print(df)

## Example 2: Load all saved tyre configs

We're going to load all of the configurations which have been saved under the `car.tyres.front` path. This requires adding a _sub_tree_path_.

In [ ]:
# We don't have a neat wrapper for this function, so we'll use the API directly
from typing import List

config_api = canopy.openapi.ConfigApi(session.async_client)

sub_tree_path = 'tyres.front'

# Note that these are returned in order of oldest first
metadata_result: canopy.openapi.GetConfigsQueryResult = await config_api.config_get_configs(session.authentication.tenant_id, 'car', **canopy.defined_kwargs(
            sub_tree_path=sub_tree_path))

documents: List[canopy.openapi.CanopyDocument] = metadata_result.query_results.documents
if len(documents) == 0:
    raise canopy.NotFoundError('No config found matching the specified criteria.')

# The documents list is now a list of all of the configs that have been saved at that specific sub_tree_path.
# This might be enough for your needs, but if you want to load the objects themselves, you need to do that explicitly:

async def load_sub_tree_config(document: canopy.openapi.CanopyDocument, sub_tree_path: str):
    tenant_id = session.authentication.tenant_id
    # You MUST specify the sub_tree_path here again when loading, otherwise you get an invalid ID
    config = await canopy.load_config(session, document.document_id, tenant_id=tenant_id, sub_tree_path=sub_tree_path)
    logging.info(f'Loaded car sub-tree config: {config.config_id}')
    return config

# Just the last five tyres for demonstration purposes
all_tyres = [await load_sub_tree_config(doc, sub_tree_path) for doc in documents[-5:]]

# For each tyre, let's find out the grip factor
tyre_data_table = []
for tyre_config in all_tyres:
    tyre_data_table.append({
        'Tyre ID': tyre_config.config_id,
        'Path': tyre_config.raw_data['path'],
        'Grip Factor': tyre_config.raw_data['definition']['rGripFactor']
    })

# Display the data table
import pandas as pd
df = pd.DataFrame(tyre_data_table)
print(df)

## Example 3: Search metadata and custom properties using a filter

Canopy enables you to set custom properties on top-level configs. They can be used in a filter to select specific configurations.
However, note that filtering to custom properties is only available on top-level configs and cannot be used with a sub-tree path in this way.

In [ ]:
# We don't have a neat wrapper for this function, so we'll use the API directly
from typing import List

config_api = canopy.openapi.ConfigApi(session.async_client)

filter = canopy.create_list_filter(
    session,
    is_study=False,
    custom_properties={'Repo':'python-examples'}
    )

# Note that these are returned in order of oldest first
metadata_result: canopy.openapi.GetConfigsQueryResult = await config_api.config_get_configs(session.authentication.tenant_id, 'car', **canopy.defined_kwargs(
            filter=filter.serialize()
            ))

documents: List[canopy.openapi.CanopyDocument] = metadata_result.query_results.documents
if len(documents) == 0:
    raise canopy.NotFoundError('No config found matching the specified criteria.')

# We can load the config and check that it has the custom property
document = documents[-1]
config = await canopy.load_config(session, document.document_id, tenant_id=session.authentication.tenant_id)
print(f'Loaded config: {config.config_id} with Repo={document.properties["Repo"]}')

## Example 4: Searching for a specific value in the config data

On the platform, you can search for configs which have a specific value, e.g. chassis models with a running mass of > 3000 kg.
We can do this too through the API by manually creating our own list filter instead of using the `create_list_filter` helper.

This is about as direct as we can get into calling directly into the API

In [ ]:
config_api = canopy.openapi.ConfigApi(session.async_client)

# Find chassis configs which meet the criteria
sub_tree_path = 'chassis'

conditions: List[canopy.openapi.ListFilterCondition] = [
    canopy.openapi.ListFilterCondition(
            source='config',
            name='carRunningMass.mCar',
            operator='greaterThan',
            value=3000
        )
]

group = canopy.openapi.ListFilterGroup(
        operator='and',
        conditions=conditions)

filter = canopy.SerializableValue(
        session,
        canopy.openapi.ListFilter(
            items_per_page=0,
            order_by_property='modifiedDate',
            order_by_descending=False,
            query=group)
        )

metadata_result: canopy.openapi.GetConfigsQueryResult = await config_api.config_get_configs(
    session.authentication.tenant_id,
    'car',
    **canopy.defined_kwargs(
        filter=filter.serialize(),
        sub_tree_path=sub_tree_path))

documents: List[canopy.openapi.CanopyDocument] = metadata_result.query_results.documents
if len(documents) == 0:
    raise canopy.NotFoundError('No config found matching the specified criteria.')

# Print out the masses of the matching chassis configs
for document in documents:
    config = await canopy.load_config(session, document.document_id, tenant_id=session.authentication.tenant_id, sub_tree_path=sub_tree_path)
    mass = config.raw_data['definition']['carRunningMass']['mCar']
    print(f'Chassis Config ID: {config.config_id}, Mass: {mass} kg')

## Example 5: Searching in a worksheet

Canopy configs can be moved into worksheets, which takes them out of the normal list of configs to simplify searching and minimise scrolling of configurations. We generally recommend running studies through worksheets.

In order to search in a worksheet, you need to know the name or the ID of the worksheet. We'll use a prompt to get either the name or id of the worksheet, but you may optionally choose to hard-code this.

### Load the Worksheet

See also "loading_worksheet_study_data".

In [ ]:
import re

worksheet_name_or_id = input("Worksheet name or ID: ")

worksheet_id = None
if re.match('^[0-9a-f]{32}$', worksheet_name_or_id):
    worksheet_id = worksheet_name_or_id
else:
    worksheet_name = worksheet_name_or_id
    worksheet = await canopy.find_config(session, 'worksheet', worksheet_name)
    worksheet_id = worksheet.config_id

logging.info(f'Using worksheet ID: {worksheet_id}')

### Search in the worksheet for a car configuration

We want to retrieve all of the cars in the worksheet, so we need to use our direct API calls.

In [ ]:
# We don't have a neat wrapper for this function, so we'll use the API directly
from typing import List

config_api = canopy.openapi.ConfigApi(session.async_client)

# Define a filter which includes the parent_worksheet_id
filter = canopy.create_list_filter(
    session,
    is_study=False,
    parent_worksheet_id=worksheet_id,
    )

# Note that these are returned in order of oldest first
metadata_result: canopy.openapi.GetConfigsQueryResult = await config_api.config_get_configs(session.authentication.tenant_id, 'car',
        **canopy.defined_kwargs(
            filter=filter.serialize()))

documents: List[canopy.openapi.CanopyDocument] = metadata_result.query_results.documents
if len(documents) == 0:
    raise canopy.NotFoundError('No config found matching the specified criteria.')

async def load_car_from_document(document: canopy.openapi.CanopyDocument):
    tenant_id = session.authentication.tenant_id
    car = await canopy.load_config(session, document.document_id, tenant_id=tenant_id)
    logging.info(f'Loaded car: {car.document.name} ({car.config_id})')
    return car

all_cars = [await load_car_from_document(doc) for doc in documents]

# Now you can do what you like with the loaded cars